# Mind Map Agent Test (Stage 12)

Full pipeline:
1. Load ChromaDB index (retriever)
2. Retrieve relevant chunks for a topic
3. Build a hierarchical mind map via `MindMapAgent`

Output includes:
- **title** — central topic
- **nodes** — nested branches and sub-concepts
- **to_text()** — textual tree visualization

**Prerequisites:**
- Index already created (`01_test_rag_pipeline.ipynb`, Part A)
- `.env` with `HUGGINGFACE_API_TOKEN` (Stage 7)
- `uv sync`

## Configuration

In [2]:
import json
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VECTOR_DB_PATH = ROOT / "data/vector_db"

TOP_K = 5
QUESTION = "Organize the main concepts of the Transformer architecture"

token = os.getenv("HUGGINGFACE_API_TOKEN") or os.getenv("HF_TOKEN")

print(f"Project root: {ROOT}")
print(f"Vector DB: {VECTOR_DB_PATH}")
print(f"Default topic: {QUESTION}")
print(f"HF token: {'configured' if token else 'NOT configured'}")

Project root: d:\Github\research-workflow-agent
Vector DB: d:\Github\research-workflow-agent\data\vector_db
Default topic: Organize the main concepts of the Transformer architecture
HF token: configured


## 1. Load retriever

In [3]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()

chunk_count = store.collection.count()
print(f"Collection: {store.collection_name} ({chunk_count} chunks)")

if chunk_count == 0:
    raise RuntimeError("Empty index. Run notebook 01_test_rag_pipeline (Part A) first.")

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

d:\Github\research-workflow-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: book_research (52 chunks)


## 2. Retrieve context (RAG)

In [4]:
context = retriever.invoke(QUESTION)

print(f"Topic: {QUESTION}")
print(f"Chunks retrieved: {len(context)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6812.60it/s]


Topic: Organize the main concepts of the Transformer architecture
Chunks retrieved: 5


## 3. Run Mind Map Agent

In [5]:
from agents.mindmap_agent import MindMapAgent

print("Calling Mind Map Agent (Hugging Face API)...")
agent = MindMapAgent()
result = agent.run(QUESTION, context)

Calling Mind Map Agent (Hugging Face API)...


In [6]:
print("Structured JSON output:")
print(result.to_json())

Structured JSON output:
{
  "title": "Transformer Architecture",
  "nodes": [
    {
      "label": "Model Components",
      "children": [
        {
          "label": "Encoder",
          "children": [
            {
              "label": "Stacked Layers",
              "children": [
                {
                  "label": "Multi-Head Self-Attention",
                  "children": []
                },
                {
                  "label": "Position-Wise Feed-Forward Network",
                  "children": []
                }
              ]
            },
            {
              "label": "Residual Connections and Layer Normalization",
              "children": []
            }
          ]
        },
        {
          "label": "Decoder",
          "children": [
            {
              "label": "Stacked Layers",
              "children": [
                {
                  "label": "Multi-Head Self-Attention",
                  "children": []
                },

In [7]:
print("=" * 60)
print("TEXTUAL MIND MAP")
print("=" * 60)
print()
print(result.to_text())

TEXTUAL MIND MAP

Transformer Architecture
├── Model Components
│   ├── Encoder
│   │   ├── Stacked Layers
│   │   │   ├── Multi-Head Self-Attention
│   │   │   └── Position-Wise Feed-Forward Network
│   │   └── Residual Connections and Layer Normalization
│   └── Decoder
│       └── Stacked Layers
│           ├── Multi-Head Self-Attention
│           └── Position-Wise Feed-Forward Network
├── Architecture Variations
│   ├── Hyperparameters
│   │   ├── d_model
│   │   ├── d_ff
│   │   ├── h
│   │   ├── d_k
│   │   ├── d_v
│   │   ├── Pdrop
│   │   └── ϵ_ls
│   └── Training Settings
│       ├── Batch Size
│       ├── Number of Training Steps
│       └── Learning Rate
└── Advantages and Applications
    ├── Parallelization
    ├── Improved Translation Quality
    └── Textual Entailment and Sentence Representations


## 4. Test another topic

Change `another_topic` and run the cells below.

In [8]:
another_topic = "How attention replaces recurrence in sequence models"

context2 = retriever.invoke(another_topic)
result2 = agent.run(another_topic, context2)

print(result2.to_text())
print()
print(json.dumps(result2.to_dict(), indent=2, ensure_ascii=False))

Attention in Sequence Models
├── Sequential Nature
│   ├── Sequential Computation Limitations
│   │   ├── Memory Constraints
│   │   ├── Factorization Tricks
│   │   └── Conditional Computation
│   └── Attention Mechanisms
│       ├── Global Dependencies
│       ├── Self-Attention
│       │   ├── Intra-Attention
│       │   └── Multi-Head Attention
│       └── End-to-End Memory Networks
├── Transformer Model
│   ├── Architecture
│   └── Advantages
│       ├── Computational Efficiency
│       └── Model Performance
└── Comparison with Recurrent Models
    ├── Computational Complexity
    └── Parallelization

{
  "title": "Attention in Sequence Models",
  "nodes": [
    {
      "label": "Sequential Nature",
      "children": [
        {
          "label": "Sequential Computation Limitations",
          "children": [
            {
              "label": "Memory Constraints",
              "children": []
            },
            {
              "label": "Factorization Tricks",
           